In [10]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from datetime import datetime
import pydicom
from pathlib import Path
import matplotlib.pyplot as plt

def seleccionar_fichero():
    ruta = filedialog.askopenfilename(  # Abre un cuadro de diálogo para seleccionar un archivo de imagen SPECT.
        title="Seleccionar fichero SPECT",
        filetypes=[("Todos los archivos", "*.*")]
    )
    if ruta:
        entry_fichero.delete(0, tk.END) # Limpia cualquier ruta de archivo que estuviera allí antes.
        entry_fichero.insert(0, ruta) # Inserta el contenido de la variable ruta

def leer_dicom(ruta):
    """Lee el archivo DICOM y devuelve información básica."""
    try:
        ds = pydicom.dcmread(ruta) # (DICOM Read) abre el archivo, lo analiza y carga todos sus datos (metadatos e imagen) en una variable (ds).

        info = [
            f"Nombre del paciente: {getattr(ds, 'PatientName', 'Desconocido')}",
            f"ID del paciente: {getattr(ds, 'PatientID', 'N/A')}",
            f"Modalidad: {getattr(ds, 'Modality', 'N/A')}",
            f"Dimensiones: {ds.Rows} x {ds.Columns}",
        ]
        return "\n".join(info)
    
    except Exception as e:
        messagebox.showerror("Error al leer DICOM", f"No se pudo leer el archivo:\n{e}")
        return None

def guardar_datos():
    """Valida y muestra los datos introducidos."""
    fichero = entry_fichero.get()
    fecha = entry_fecha.get()
    actividad = entry_actividad.get()

    # Validar que todos los campos están completos
    if not fichero or not fecha or not actividad:
        messagebox.showwarning("Campos incompletos", "Por favor, complete todos los campos.")
        return

    # Validar formato de fecha
    try:
        datetime.strptime(fecha, "%d/%m/%Y")
    except ValueError:
        messagebox.showerror("Error de formato", "La fecha debe tener el formato dd/mm/aaaa.")
        return
    
    # Leer el archivo DICOM
    info_dicom = leer_dicom(fichero)
    if not info_dicom:
        return

    # Mostrar resumen
    resumen = (
        f"Datos introducidos correctamente\n\n"
        f"Fichero: {fichero}\n"
        f"Fecha: {fecha}\n"
        f"Actividad: {actividad} MBq\n\n"
        f"Información del DICOM:\n{info_dicom}"
    )

    messagebox.showinfo("Resumen", resumen)

# Crear ventana principal
root = tk.Tk()
root.title("Ingreso de datos SPECT")
root.geometry("700x250")
root.resizable(False, False)

# Estilo
style = ttk.Style()
style.configure("TLabel", font=("Calibri", 11))
style.configure("TButton", font=("Calibri", 10))
style.configure("TEntry", font=("Calibri", 10))

# Etiquetas y campos
ttk.Label(root, text="Fichero SPECT:").grid(row=0, column=0, padx=10, pady=10, sticky="e") # grid permite colocar cada widget en una celda
entry_fichero = ttk.Entry(root, width=40)
entry_fichero.grid(row=0, column=1, padx=5, pady=10)
ttk.Button(root, text="Examinar...", command=seleccionar_fichero).grid(row=0, column=2, padx=5, pady=10)

ttk.Label(root, text="Fecha de adquisición (dd/mm/aaaa):").grid(row=1, column=0, padx=10, pady=10, sticky="e") # e = este = derecha
entry_fecha = ttk.Entry(root, width=20)
entry_fecha.grid(row=1, column=1, padx=5, pady=10, sticky="w") # w = oeste = izquierda

ttk.Label(root, text="Actividad (MBq):").grid(row=2, column=0, padx=10, pady=10, sticky="e")
entry_actividad = ttk.Entry(root, width=20)
entry_actividad.grid(row=2, column=1, padx=5, pady=10, sticky="w")

# Botón de guardar
ttk.Button(root, text="Guardar datos", command=guardar_datos).grid(row=3, column=1, pady=20)


# Ejecutar interfaz
root.mainloop()


In [ ]:
import napari
import pydicom
import numpy as np

# --- Cargar el DICOM ---
file_path = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\Tomo4FOV_Lu177peak_IRACSC001_DS.dcm"
ds = pydicom.dcmread(file_path)
image = ds.pixel_array.astype(float)

# --- Lanzar visor ---
viewer = napari.view_image(image, name="Lu-177 Tomografía", colormap='gray', contrast_limits=[0, np.max(image)])
napari.run()

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\gervi\\OneDrive - Universidad Complutense de Madrid (UCM)\\MASTER\\SEGUNDO CUATRI\\TFM\\Imagenes prueba\\CT'

In [3]:
import os
import numpy as np
import pydicom
import napari

# --- Carpeta con los DICOM ---
folder = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\CT"

# --- Leer todos los archivos .dcm y ordenarlos (opcional: por nombre o por SliceLocation) ---
dicom_files = [f for f in os.listdir(folder) if f.endswith(".dcm")]
dicom_files.sort()  # si los nombres siguen el orden de los cortes

# --- Cargar los DICOM y extraer arrays ---
slices = []
for f in dicom_files:
    ds = pydicom.dcmread(os.path.join(folder, f))
    slices.append(ds.pixel_array.astype(float))

# --- Apilar en un array 3D (Z, Y, X) ---
image_3d = np.stack(slices, axis=0)

# --- Lanzar visor 3D en Napari ---
viewer = napari.view_image(
    image_3d, 
    name="Lu-177 Tomografía", 
    colormap='gray', 
    contrast_limits=[0, np.max(image_3d)]
)
napari.run()

C:\Users\gervi\AppData\Local\Temp\ipykernel_19544\3478411735.py:23: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(


In [ ]:
import SimpleITK as sitk
import numpy as np
import json

# --------------------------
# CREAR UNA MÁSCARA ESFÉRICA
# --------------------------
def create_sphere_mask(ct_image, center_mm, radius_mm):
    size = ct_image.GetSize()
    spacing = np.array(ct_image.GetSpacing())
    origin = np.array(ct_image.GetOrigin())
    direction = np.array(ct_image.GetDirection()).reshape(3,3)º

    # matriz de voxels
    mask = np.zeros(size[::-1], dtype=np.uint8)  # SimpleITK usa z,y,x
    coords = np.indices(size).reshape(3, -1).T  # Nx3 (i,j,k)

    # transformar índices a coordenadas físicas
    phys_coords = origin + (coords @ direction.T) * spacing

    # distancia al centro de la esfera
    dist = np.linalg.norm(phys_coords - np.array(center_mm), axis=1)
    inside = dist <= radius_mm

    mask.flat[inside] = 1

    mask_img = sitk.GetImageFromArray(mask)
    mask_img.CopyInformation(ct_image)
    return mask_img

# --------------------------
# CREAR TODAS LAS MÁSCARAS
# --------------------------
def generate_all_sphere_masks(ct_path, json_path):
    ct = sitk.ReadImage(ct_path)

    with open(json_path, "r") as f:
        centers = json.load(f)

    sphere_sizes = {
        "sphere_10mm": 10/2,
        "sphere_13mm": 13/2,
        "sphere_17mm": 17/2,
        "sphere_22mm": 22/2,
        "sphere_28mm": 28/2,
        "sphere_37mm": 37/2
    }

    masks = {}
    for key, radius in sphere_sizes.items():
        center = centers[key]
        print(f"Generando máscara: {key} (radio={radius} mm, centro={center})")
        mask = create_sphere_mask(ct, center, radius)
        masks[key] = mask

    return masks
